# Lecture 05: Attention and the Transformer

CS40008.01 · Baojian Zhou · Fudan University · October 10, 2026

**How does attention select useful context, and how does it lead to the Transformer?**

Predict each result before running its cell. E01–E05 are **ungraded classroom practices**, in slide order, with the same expectations for all students. This notebook runs offline on CPU with the course environment (`uv sync`). It uses PyTorch, included through the existing environment, and no downloaded datasets, tokenizers, or model weights. All sequences and measurements below are explicitly **toy examples**.

| Exercise | Time | Student response |
| --- | --- | --- |
| E01 | 5 min | Head split/merge shapes; batched heads versus a per-head loop |
| E02 | 5 min | Predict a permutation experiment with and without positions |
| E03 | 4 min | One LayerNorm and residual update; the normalization axis |
| E04 | 5 min | Parameter count of a block and of the full model |
| E05 | 6 min | Causality, gradients, and a tiny-batch fitting run |

We build a **small decoder-only teaching model**, not the whole 2017 encoder–decoder system. The cells include runnable explanations after the prediction prompts. To experiment, edit a personal copy under `workspace/`. The course Notebook launcher preserves existing copies.

In [ ]:
import copy
import math

import torch
import torch.nn.functional as F
from torch import nn

print('PyTorch:', torch.__version__, '| device: CPU')

## From recurrent context to learned alignment

An RNN carries information through successive hidden states. LSTM gates help retain it, while each state still depends on its predecessor. A fixed-context encoder–decoder passes one source summary to the decoder. Attention keeps several source states available and selects a weighted context for each target step.

With decoder query $q_t$ and source state $h_s$, compute a score $e_{t,s}=a(q_t,h_s)$, normalize over source positions, and combine:

$$\alpha_{t,s}=\frac{\exp(e_{t,s})}{\sum_j\exp(e_{t,j})},\qquad c_t=\sum_s\alpha_{t,s}h_s.$$

[Bahdanau et al. (2015), §3](https://arxiv.org/abs/1409.0473) learn an additive alignment network using the previous decoder state. [Luong et al. (2015), §3.1](https://aclanthology.org/D15-1166/) compare dot, general, and concat scores using the current decoder state. Their recurrent update conventions differ. The shared mechanism is score, normalize, and combine.

**Worked example:** $q=(1,0)$, $h_1=(0,1)$, $h_2=(1,0)$, with an unscaled dot score. Scores are $(0,1)$; softmax weights are approximately $(0.2689,0.7311)$; context is $(0.7311,0.2689)$. These invented states serve as both keys and values. The Transformer head below adds Q/K/V projections and scaling.


In [ ]:
alignment_query = torch.tensor([1., 0.], dtype=torch.float64)
alignment_states = torch.tensor([[0., 1.], [1., 0.]], dtype=torch.float64)
alignment_scores = alignment_states @ alignment_query
alignment_weights = alignment_scores.softmax(dim=0)
alignment_context = alignment_weights @ alignment_states
print('scores:', alignment_scores.tolist())
print('weights:', alignment_weights.tolist(), '| context:', alignment_context.tolist())


## One scaled attention head, step by step

Queries score keys; the resulting weights combine values. In self-attention,
learned projections obtain all three from the same sequence of states.
No earlier attention implementation is assumed here.

Use already-projected toy vectors $q=(\sqrt{2},0)$, keys
$(1,0),(0,1),(1,1)$, and values $(1,0),(0,2),(2,1)$.
Divide the dot scores by $\sqrt{d_k}=\sqrt{2}$, normalize over the three
keys, and form the weighted output. These invented coordinates carry no
assigned word meanings. Source: [Vaswani et al., §3.2.1](https://arxiv.org/abs/1706.03762).

In [ ]:
head_query = torch.tensor([math.sqrt(2), 0.], dtype=torch.float64)
head_keys = torch.tensor([[1., 0.], [0., 1.], [1., 1.]], dtype=torch.float64,
                         requires_grad=True)
head_values = torch.tensor([[1., 0.], [0., 2.], [2., 1.]], dtype=torch.float64,
                           requires_grad=True)
head_scores = head_keys @ head_query / math.sqrt(head_query.numel())
head_weights = head_scores.softmax(dim=-1)
head_output = head_weights @ head_values
print('scaled scores:', head_scores.detach().tolist())
print('weights:', head_weights.detach().tolist())
print('output:', head_output.detach().tolist())

### Apply causal visibility before normalization

Put the query at input slot 1: slots 0 and 1 are allowed, while slot 2 is
future. A next-token model can read its current input because the target
is the following token. Replace the future score with $-\infty$, then
recompute softmax. The allowed weights become approximately
$(0.7311,0.2689)$ and still sum to one.

The following checks change a future value and differentiate through the
key/value vectors. The unmasked computation is a negative control. Merely
zeroing a weight after the original softmax does not renormalize the row.
Every query needs at least one allowed key; an entirely masked row has an
undefined softmax. Source: [Vaswani et al., §3.2.3](https://arxiv.org/abs/1706.03762).

In [ ]:
head_allowed = torch.tensor([True, True, False])
head_masked_weights = head_scores.masked_fill(~head_allowed, -torch.inf).softmax(-1)
head_masked_output = head_masked_weights @ head_values
head_changed_values = head_values.detach().clone()
head_changed_values[2] = torch.tensor([200., -100.], dtype=torch.float64)
head_masked_changed = head_masked_weights.detach() @ head_changed_values
head_unmasked_changed = head_weights.detach() @ head_changed_values
head_key_grad, head_value_grad = torch.autograd.grad(
    head_masked_output.sum(), (head_keys, head_values)
)
head_wrong_weights = head_weights.detach() * head_allowed
print('masked weights:', head_masked_weights.detach().tolist())
print('masked output:', head_masked_output.detach().tolist())
print('future-value change, masked:', (head_masked_changed - head_masked_output).detach().tolist())
print('future-value change, unmasked:', (head_unmasked_changed - head_output).detach().tolist())
print('future key/value derivatives:', head_key_grad[2].tolist(), head_value_grad[2].tolist())
print('wrong post-softmax masking: row sum =', head_wrong_weights.sum().item())

## The same two sentences as Lecture 04

We reuse Lecture 04's invented corpus and vocabulary IDs:

```
BOS red  key opens  EOS
BOS blue key closes EOS
```

A decoder language model reads the first four tokens and predicts the **next** token at every position: inputs `X = documents[:, :-1]`, targets `y = documents[:, 1:]`. Position $t$ of the output predicts token $t+1$.

Both sentences start with the identical prefix `BOS`, followed once by `red` and once by `blue`. No model can predict both. The best achievable probability is $1/2$ for each, so the two BOS targets together cost at least $2\log 2$ nats. The **minimum possible mean training loss** is $2\log 2/8=\log(2)/4\approx0.1733$. The other six transitions are deterministic in this toy batch.

In [ ]:
vocabulary = ['BOS', 'red', 'blue', 'key', 'opens', 'closes', 'EOS']
documents = torch.tensor([[0, 1, 3, 4, 6], [0, 2, 3, 5, 6]])
X, y = documents[:, :-1], documents[:, 1:]
B, T = X.shape
V, max_length, d, n_heads, d_ff, n_layers = len(vocabulary), 4, 16, 4, 32, 2
loss_floor = math.log(2) / 4
for inputs, targets in zip(X.tolist(), y.tolist()):
    print([f'{vocabulary[a]} -> {vocabulary[b]}' for a, b in zip(inputs, targets)])
print('minimum mean training loss:', loss_floor)

## Multiple heads: split, attend, merge, project

The preceding worked example introduced one causal head. With model width $d=16$ and $h=4$ heads, each head has width $d_h=d/h=4$:

1. project the whole input once: $Q=XW_Q^\top$, $K=XW_K^\top$, $V=XW_V^\top$, each `(B, T, d)`;
2. **split** the feature axis into heads, `(B, T, h, d_h)`, then **transpose** to `(B, h, T, d_h)`;
3. run causal scaled dot-product attention in every head in parallel;
4. **merge**: transpose back to `(B, T, h, d_h)`, reshape to `(B, T, d)`;
5. apply the output projection $W_O$.

At fixed $d$, the four $d\times d$ projection matrices have the same size for any valid $h$. Different heads have different learned projections; that alone does not guarantee distinct linguistic roles. Source: [Vaswani et al. (2017), §3.2.2](https://arxiv.org/abs/1706.03762). This notebook uses bias-free projections. Here $W_Q,W_K,W_V$ denote PyTorch’s stored `(out_features, in_features)` weights, so the formula includes a transpose; the slides use mathematical input-to-output matrices.

## E01 · Trace multi-head shapes

**5 minutes.** With $B=2$, $T=4$, $d=16$, and $h=4$, write the shapes after projection, split and transpose, the score tensor, the merged heads, and $W_O$.

Here $h=T=d_h=4$. Which mistake can **not** be caught by checking shapes alone? Then compare the batched implementation with an independent loop that slices each head's rows of $W_Q,W_K,W_V$ and concatenates the head outputs.

In [ ]:
def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
    elif isinstance(module, nn.LayerNorm):
        nn.init.ones_(module.weight)
        nn.init.zeros_(module.bias)


def split_heads(x, n_heads):
    batch, length, width = x.shape
    if width % n_heads:
        raise ValueError('model width must be divisible by the number of heads')
    return x.reshape(batch, length, n_heads, width // n_heads).transpose(1, 2)


def merge_heads(x):
    batch, heads, length, head_width = x.shape
    return x.transpose(1, 2).reshape(batch, length, heads * head_width)


class MultiHeadAttention(nn.Module):
    def __init__(self, width, n_heads):
        super().__init__()
        if width % n_heads:
            raise ValueError('model width must be divisible by the number of heads')
        self.n_heads = n_heads
        self.query = nn.Linear(width, width, bias=False)
        self.key = nn.Linear(width, width, bias=False)
        self.value = nn.Linear(width, width, bias=False)
        self.output = nn.Linear(width, width, bias=False)

    def forward(self, x, causal=True):
        q, k, v = (split_heads(layer(x), self.n_heads)
                   for layer in (self.query, self.key, self.value))
        scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
        if causal:
            allowed = torch.ones(x.shape[1], x.shape[1], dtype=torch.bool, device=x.device).tril()
            scores = scores.masked_fill(~allowed, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        return self.output(merge_heads(weights @ v)), weights

In [ ]:
def heads_loop_reference(attention, x, causal=True):
    # Independent reference: slice each head's rows and concatenate head outputs.
    head_width = x.shape[-1] // attention.n_heads
    future = torch.ones(x.shape[1], x.shape[1], dtype=torch.bool).triu(1)
    outputs = []
    for head in range(attention.n_heads):
        rows = slice(head * head_width, (head + 1) * head_width)
        q = x @ attention.query.weight[rows].T
        k = x @ attention.key.weight[rows].T
        v = x @ attention.value.weight[rows].T
        scores = q @ k.transpose(-2, -1) / math.sqrt(head_width)
        if causal:
            scores = scores.masked_fill(future, float('-inf'))
        outputs.append(torch.softmax(scores, dim=-1) @ v)
    return torch.cat(outputs, dim=-1) @ attention.output.weight.T


torch.manual_seed(7)
attention = MultiHeadAttention(d, n_heads).double()  # default PyTorch initialization
e01_x = torch.randn(B, T, d, dtype=torch.float64, requires_grad=True)
e01_output, e01_weights = attention(e01_x)
e01_split = split_heads(attention.query(e01_x), n_heads)
e01_shapes = {
    'input': tuple(e01_x.shape), 'projected': tuple(attention.query(e01_x).shape),
    'split heads': tuple(e01_split.shape), 'scores': tuple(e01_weights.shape),
    'merged': tuple(merge_heads(e01_split).shape), 'output': tuple(e01_output.shape),
}
print(e01_shapes)

reference_x = e01_x.detach().clone().requires_grad_()
e01_reference = heads_loop_reference(attention, reference_x)
probe = torch.randn(B, T, d, dtype=torch.float64)
parameters = tuple(attention.parameters())
batched_grads = torch.autograd.grad((e01_output * probe).sum(), (e01_x, *parameters))
reference_grads = torch.autograd.grad((e01_reference * probe).sum(), (reference_x, *parameters))
torch.testing.assert_close(e01_output, e01_reference, rtol=1e-10, atol=1e-10)
for actual_grad, reference_grad in zip(batched_grads, reference_grads):
    torch.testing.assert_close(actual_grad, reference_grad, rtol=1e-10, atol=1e-10)
e01_forward_error = (e01_output - e01_reference).abs().max().item()
e01_gradient_error = max((a - b).abs().max().item() for a, b in zip(batched_grads, reference_grads))
print('max forward / gradient error:', e01_forward_error, e01_gradient_error)

In [ ]:
def wrong_split(x, n_heads):  # reshape directly to (B, h, T, d_h): no transpose
    batch, length, width = x.shape
    return x.reshape(batch, n_heads, length, width // n_heads)


def wrong_merge(x):  # reshape (B, h, T, d_h) directly to (B, T, d)
    batch, heads, length, head_width = x.shape
    return x.reshape(batch, length, heads * head_width)


def wrong_attention(x):
    q, k, v = (wrong_split(layer(x), n_heads)
               for layer in (attention.query, attention.key, attention.value))
    future = torch.ones(T, T, dtype=torch.bool).triu(1)
    scores = (q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])).masked_fill(future, float('-inf'))
    return attention.output(wrong_merge(torch.softmax(scores, dim=-1) @ v))


with torch.no_grad():
    wrong_output = wrong_attention(e01_x)
    changed_x = e01_x.detach().clone()
    changed_x[:, 1] += 1.0  # change token 1 only
    correct_change = (attention(changed_x)[0] - e01_output)[:, 3].abs().max().item()
    wrong_change = (wrong_attention(changed_x) - wrong_output)[:, 3].abs().max().item()
e01_wrong_shape = tuple(wrong_output.shape)
e01_wrong_error = (wrong_output - e01_output).abs().max().item()
print('wrong version shape:', e01_wrong_shape, '| max difference:', e01_wrong_error)
print('token 1 change seen at position 3, correct / wrong:', correct_change, wrong_change)

try:
    MultiHeadAttention(10, n_heads)
except ValueError as error:
    e01_divisibility_error = str(error)
    print('Expected failure:', e01_divisibility_error)

**Check:** input `(2, 4, 16)`; projection `(2, 4, 16)`; split and transpose `(2, 4, 4, 4)` $=(B,h,T,d_h)$; scores `(2, 4, 4, 4)` $=(B,h,T,T)$; merged `(2, 4, 16)`; output `(2, 4, 16)`. The batched output and every gradient (input, $W_Q$, $W_K$, $W_V$, $W_O$) match the per-head loop within the stated float64 tolerance, $10^{-10}$.

Reshaping directly to `(B, h, T, d_h)` has the **same shape** but a different meaning. Row-major order puts token $i$'s whole vector into "head" $i$, and its four feature chunks become the "positions". The wrong version then mixes chunks of one token instead of mixing tokens: changing token 1 leaves its position-3 output unchanged, while the correct causal model does change. Because $h=T=d_h$ here, only a value check detects this bug. A width that is not divisible by the number of heads is rejected.

## Where does order come from?

Without a mask or positional information, dot-product self-attention is **permutation equivariant**: permuting the input rows permutes the output rows in the same way. A **causal mask** already imposes an ordering constraint; to isolate the effect of positions, the next experiment is **unmasked**.

- **Sinusoidal (historical, original Transformer):** $PE(p,2i)=\sin(p/10000^{2i/d})$ and $PE(p,2i+1)=\cos(p/10000^{2i/d})$, added to the input at fixed slots. [Vaswani et al. (2017), §3.5](https://arxiv.org/abs/1706.03762).
- **Learned absolute positions (our baseline):** an `nn.Embedding(max_length, d)` table trained jointly with the token table. It cannot represent positions beyond `max_length`.
- **RoPE (optional contrast):** rotate each query/key feature pair by a position-dependent angle, so that their dot product depends on the relative offset. [Su et al., RoFormer, §3](https://arxiv.org/abs/2104.09864).

## E02 · A permutation experiment

**5 minutes.** Four tokens `bank of the river` are represented by the rows of the $4\times4$ identity matrix $X$. Use $Q=K=V$ equal to the input, **no mask**, and scale $\sqrt{4}=2$. Permute the rows with `[3, 1, 2, 0]` (swap `bank` and `river`), run attention, and undo the permutation on the output.

Predict: (a) without positions, does the restored output equal the original output? (b) With sinusoidal positions added at the **fixed** slots 0–3 after permuting, i.e. $Q=K=V=X_\pi+PE$?

In [ ]:
def sinusoidal_positions(length, width, dtype=torch.float64):
    if width % 2:
        raise ValueError('sinusoidal positions need an even width')
    positions = torch.arange(length, dtype=dtype)[:, None]
    frequencies = 10000 ** (-torch.arange(0, width, 2, dtype=dtype) / width)
    table = torch.zeros(length, width, dtype=dtype)
    table[:, 0::2] = torch.sin(positions * frequencies)
    table[:, 1::2] = torch.cos(positions * frequencies)
    return table


def identity_attention(z):  # unmasked, with Q = K = V = z
    return torch.softmax(z @ z.T / math.sqrt(z.shape[-1]), dim=-1) @ z


e02_tokens = ['bank', 'of', 'the', 'river']
e02_x = torch.eye(4, dtype=torch.float64)
e02_permutation = torch.tensor([3, 1, 2, 0])
e02_inverse = torch.argsort(e02_permutation)
e02_pe = sinusoidal_positions(4, 4)

e02_plain = identity_attention(e02_x)
e02_plain_restored = identity_attention(e02_x[e02_permutation])[e02_inverse]
e02_positioned = identity_attention(e02_x + e02_pe)
e02_positioned_restored = identity_attention(e02_x[e02_permutation] + e02_pe)[e02_inverse]
e02_plain_error = (e02_plain_restored - e02_plain).abs().max().item()
e02_positioned_error = (e02_positioned_restored - e02_positioned).abs().max().item()
e02_fixture = {
    'tokens': e02_tokens, 'permutation': e02_permutation.tolist(),
    'positions': e02_pe.tolist(), 'plain_output': e02_plain.tolist(),
    'positioned_output': e02_positioned.tolist(),
    'positioned_restored': e02_positioned_restored.tolist(),
    'plain_error': e02_plain_error, 'positioned_error': e02_positioned_error,
}
print('bank row without positions:', [round(v, 4) for v in e02_plain[0].tolist()])
print('restored error without / with positions:', e02_plain_error, e02_positioned_error)

**Check:** (a) yes. Every row of $XX^\top/2$ has $0.5$ on the diagonal and $0$ elsewhere, so `bank` receives $e^{0.5}/(e^{0.5}+3)\approx0.3547$ from itself and $\approx0.2151$ from each other token; the restored error is $0$ up to rounding. (b) No: `bank` now sits in slot 3 and receives $PE(3)$ instead of $PE(0)$, so its scores and output change. The restored error is clearly nonzero.

This demonstrates what positions add to **unmasked** attention. A decoder's causal mask already distinguishes earlier from later positions, and our model below adds **learned** positions. The sinusoidal table is the historical contrast; RoPE instead changes query/key dot products.

## Residual paths and LayerNorm

A **residual** connection adds a sublayer's output to its input: $x\leftarrow x+F(x)$. **LayerNorm** normalizes each token vector over its **feature axis** (the last axis, width $d$), independently of other positions and batch items, then applies a learned scale $\gamma$ and shift $\beta$:

$$\mathrm{LN}(x)=\gamma\odot\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta,\quad \mu,\sigma^2 \text{ over the } d \text{ features}.$$

- **Post-LN** (original Transformer): $x\leftarrow\mathrm{LN}(x+F(x))$.
- **Pre-LN** (our baseline): $x\leftarrow x+F(\mathrm{LN}(x))$.

Residual paths and normalization help optimization; they do not guarantee the absence of vanishing or exploding gradients. [Ba et al. (2016)](https://arxiv.org/abs/1607.06450) define LayerNorm; [Xiong et al. (2020)](https://proceedings.mlr.press/v119/xiong20b.html) analyze pre- and post-LN placement. The **position-wise FFN** $W_2\,\mathrm{ReLU}(W_1x)$ is applied to each position separately ([Vaswani et al., §3.3](https://arxiv.org/abs/1706.03762)); we omit its biases.

## E03 · One residual update and the normalization axis

**4 minutes.** (a) Apply LayerNorm with $\gamma=1$, $\beta=0$, and $\epsilon=10^{-5}$ to $x=(1,3)$. (b) A sublayer returns $F(x)=(2,-1)$: what is $x+F(x)$? (c) For hidden states `(B, T, d)`, over which axis are the mean and variance computed? 

**After the FFN explanation:** change the hidden state at position 1 only. Which output positions change for the FFN, and which for causal attention?

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, width, hidden_width):
        super().__init__()
        self.up = nn.Linear(width, hidden_width, bias=False)
        self.down = nn.Linear(hidden_width, width, bias=False)

    def forward(self, x):
        return self.down(torch.relu(self.up(x)))


class Block(nn.Module):
    def __init__(self, width, n_heads, hidden_width, norm_first=True):
        super().__init__()
        self.norm_first = norm_first
        self.attention_norm = nn.LayerNorm(width, eps=1e-5)
        self.attention = MultiHeadAttention(width, n_heads)
        self.ffn_norm = nn.LayerNorm(width, eps=1e-5)
        self.ffn = FeedForward(width, hidden_width)

    def forward(self, x, causal=True):
        if self.norm_first:  # pre-LN: x + F(LN(x))
            x = x + self.attention(self.attention_norm(x), causal)[0]
            return x + self.ffn(self.ffn_norm(x))
        x = self.attention_norm(x + self.attention(x, causal)[0])  # post-LN: LN(x + F(x))
        return self.ffn_norm(x + self.ffn(x))


e03_vector = torch.tensor([1., 3.], dtype=torch.float64)
e03_normalized = F.layer_norm(e03_vector, (2,), eps=1e-5)
e03_residual = e03_vector + torch.tensor([2., -1.], dtype=torch.float64)
print('LayerNorm([1, 3]):', e03_normalized.tolist(), '| x + F(x):', e03_residual.tolist())

torch.manual_seed(7)
e03_block = Block(d, n_heads, d_ff).double()
e03_h = 3 * torch.randn(B, T, d, dtype=torch.float64) + 5
changed_h = e03_h.clone()
changed_h[:, 1] += 1.0
with torch.no_grad():
    normed = e03_block.attention_norm(e03_h)
    e03_feature_means = normed.mean(dim=-1)
    e03_feature_variances = normed.var(dim=-1, unbiased=False)
    e03_ffn_change = (e03_block.ffn(changed_h) - e03_block.ffn(e03_h)).abs().amax(dim=(0, 2))
    e03_attention_change = (e03_block.attention(changed_h)[0]
                            - e03_block.attention(e03_h)[0]).abs().amax(dim=(0, 2))
print('per-token mean / variance after LayerNorm:',
      e03_feature_means.abs().max().item(), e03_feature_variances.mean().item())
print('FFN change by position:', e03_ffn_change.tolist())
print('attention change by position:', e03_attention_change.tolist())

e03_post_block = Block(d, n_heads, d_ff, norm_first=False).double()
e03_post_block.load_state_dict(e03_block.state_dict())
with torch.no_grad():
    e03_pre_output, e03_post_output = e03_block(e03_h), e03_post_block(e03_h)
e03_pre_post_difference = (e03_pre_output - e03_post_output).abs().max().item()
print('same parameters, pre-LN versus post-LN output difference:', e03_pre_post_difference)
print('mean feature value, pre-LN / post-LN output:',
      e03_pre_output.mean().item(), e03_post_output.mean().item())

**Check:** (a) $\mu=2$ and $\sigma^2=1$, so $\mathrm{LN}(x)=(-1,1)/\sqrt{1+10^{-5}}\approx(-0.999995,0.999995)$. (b) $(1,3)+(2,-1)=(3,2)$. (c) Over the **last (feature) axis** $d$: every one of the $B\times T$ token vectors gets mean $0$ and variance close to $1$, separately.

(d) The FFN changes **only position 1**; the other rows are exactly unchanged. Causal attention changes positions 1, 2, and 3, which may read position 1, but not position 0. Mixing across positions happens only in attention.

With the same parameters, pre-LN and post-LN compute different functions. Post-LN normalizes the stream after each addition, so its output features have mean near $0$ here. Pre-LN leaves the residual stream unnormalized; the offset of about $5$ passes straight through, which is why the model adds a **final LayerNorm** before the readout.

## Assembling a small decoder language model

1. token embedding + learned position embedding, both width $d=16$;
2. two pre-LN blocks, each with causal multi-head attention and a ReLU FFN ($d_\text{ff}=32$);
3. a final LayerNorm;
4. a bias-free vocabulary readout whose weight is **tied** to the token table, giving logits `(B, T, |V|)`.

There is no dropout. Linear and embedding weights start from $\mathcal N(0,0.02^2)$; LayerNorm starts at $\gamma=1$, $\beta=0$. Compared with the [original Transformer](https://arxiv.org/abs/1706.03762): no encoder or cross-attention, pre-LN rather than post-LN, learned rather than sinusoidal positions, no biases in projections or FFN, no $\sqrt{d}$ embedding scale, and tiny sizes. Section 3.4 of that paper also shares the embedding and pre-softmax weights; see [Press and Wolf (2017)](https://arxiv.org/abs/1608.05859).

## E04 · Count the parameters

**5 minutes.** With $d=16$, $d_\text{ff}=32$, $h=4$, $|V|=7$, and `max_length = 4`, count: (a) the four bias-free attention projections; (b) the two bias-free FFN matrices; (c) two LayerNorms with $\gamma$ and $\beta$; (d) one block; (e) the full two-block model, including both embedding tables and the final LayerNorm, with a tied readout. (f) How many parameters would an untied readout add? Does the number of heads change the count?

In [ ]:
class DecoderLM(nn.Module):
    def __init__(self, vocab_size=V, max_length=max_length, width=d, n_heads=n_heads,
                 hidden_width=d_ff, n_layers=n_layers, norm_first=True, tie_weights=True):
        super().__init__()
        self.max_length = max_length
        self.token_embedding = nn.Embedding(vocab_size, width)
        self.position_embedding = nn.Embedding(max_length, width)
        self.blocks = nn.ModuleList(Block(width, n_heads, hidden_width, norm_first)
                                    for _ in range(n_layers))
        self.final_norm = nn.LayerNorm(width, eps=1e-5)
        self.lm_head = nn.Linear(width, vocab_size, bias=False)
        self.apply(init_weights)
        if tie_weights:  # the readout reuses the same Parameter object
            self.lm_head.weight = self.token_embedding.weight

    def embed(self, ids):
        if ids.ndim != 2 or ids.dtype != torch.long:
            raise ValueError('ids must be a (batch, length) tensor of token IDs')
        if not 1 <= ids.shape[1] <= self.max_length:
            raise ValueError(f'sequence length must be between 1 and {self.max_length}')
        if ids.min() < 0 or ids.max() >= self.token_embedding.num_embeddings:
            raise ValueError('token ID outside the vocabulary')
        positions = torch.arange(ids.shape[1], device=ids.device)
        return self.token_embedding(ids) + self.position_embedding(positions)

    def forward_hidden(self, x, causal=True):
        for block in self.blocks:
            x = block(x, causal)
        return self.final_norm(x)

    def forward(self, ids, causal=True):
        return self.lm_head(self.forward_hidden(self.embed(ids), causal))

In [ ]:
def count_parameters(module):
    return sum(parameter.numel() for parameter in module.parameters())


torch.manual_seed(7)
model = DecoderLM()
untied = DecoderLM(tie_weights=False)
with torch.no_grad():
    untied.lm_head.weight.copy_(untied.token_embedding.weight)  # equal values, separate storage
e04_formula = {'attention': 4 * d * d, 'ffn': 2 * d * d_ff, 'layer norms': 2 * 2 * d}
e04_counts = {
    'block': count_parameters(model.blocks[0]),
    'blocks': sum(count_parameters(block) for block in model.blocks),
    'token table': model.token_embedding.weight.numel(),
    'position table': model.position_embedding.weight.numel(),
    'final norm': count_parameters(model.final_norm),
    'model tied': count_parameters(model),
    'model untied': count_parameters(untied),
}
e04_tied = model.lm_head.weight is model.token_embedding.weight
e04_copied_is_tied = untied.lm_head.weight is untied.token_embedding.weight
print(e04_formula, '| block formula total:', sum(e04_formula.values()))
print(e04_counts)
print('tied readout is the same Parameter:', e04_tied, '| copied values:', e04_copied_is_tied)

**Check:** (a) $4d^2=1024$; (b) $2d\,d_\text{ff}=1024$; (c) $2\times2d=64$; (d) **2112** per block. (e) Two blocks give 4224; token table $7\times16=112$; position table $4\times16=64$; final LayerNorm 32; total **4432**. (f) An untied readout adds another $7\times16=112$, giving **4544**. At fixed $d$, the head count does not change the projection sizes.

`model.parameters()` returns each registered `Parameter` once, so the tied weight is counted once. Tying means both modules hold the **same Parameter object** and receive one combined gradient; copying equal values into a separate matrix is not tying and still counts 4544.

## Readout, shifted loss, and three attention patterns

The final hidden states `(B, T, d)` become logits `(B, T, |V|)`. The shifted cross-entropy flattens them to `(B·T, |V|)` against `y` flattened to `(B·T,)`; all eight positions are trained **in parallel**. Generation still proceeds one new token at a time.

| Model | Self-attention mask | Training target |
| --- | --- | --- |
| Decoder LM (ours) | causal: position $t$ reads $\le t$ | next token at every position |
| BERT-style encoder | none (bidirectional) | masked input tokens ([Devlin et al., §3.1](https://aclanthology.org/N19-1423/)) |
| Encoder–decoder | encoder: none; decoder: causal | target token; decoder **cross-attention** queries read encoder keys/values |

Position $t$ attends to itself because its input token is already known; its **target** is token $t+1$.

## E05 · Verify the complete model

**6 minutes.** Predict before running: (a) the logits shape for `X`; (b) if only the input token at index 3 changes, which positions' logits may change? (c) For a loss using only logits at positions 0–2, which **per-position input states** receive nonzero gradient? (d) What happens without the causal mask?

We differentiate with respect to the summed token and position vectors produced by `model.embed`, using `forward_hidden`. Gradients to the shared embedding **table** would not answer (c): a table row also receives gradient whenever that token appears earlier. Then inspect a prepared tiny-batch fitting run against the loss floor $\log(2)/4$.

In [ ]:
torch.manual_seed(7)
model = DecoderLM()
initial_state = copy.deepcopy(model.state_dict())
changed_X = X.clone()
changed_X[:, 3] = torch.tensor([1, 2])  # replace 'opens'/'closes' by 'red'/'blue'
with torch.no_grad():
    e05_logits = model(X)
    e05_prefix_change = (model(changed_X) - e05_logits)[:, :3].abs().max().item()
    e05_last_change = (model(changed_X) - e05_logits)[:, 3].abs().max().item()
    e05_unmasked_prefix_change = (model(changed_X, causal=False)
                                  - model(X, causal=False))[:, :3].abs().max().item()
    e05_weights = model.blocks[0].attention(model.blocks[0].attention_norm(model.embed(X)))[1]


def prefix_state_gradient(causal):
    states = model.embed(X).detach().requires_grad_()
    logits = model.lm_head(model.forward_hidden(states, causal))
    loss = F.cross_entropy(logits[:, :3].reshape(-1, V), y[:, :3].reshape(-1))
    return torch.autograd.grad(loss, states)[0].abs().amax(dim=(0, 2))


e05_state_gradient = prefix_state_gradient(causal=True)
e05_unmasked_state_gradient = prefix_state_gradient(causal=False)
print('logits shape:', tuple(e05_logits.shape))
print('prefix logit change / last-position change:', e05_prefix_change, e05_last_change)
print('without mask, prefix logit change:', e05_unmasked_prefix_change)
print('max |gradient| per input position, causal:', e05_state_gradient.tolist())
print('max |gradient| per input position, unmasked:', e05_unmasked_state_gradient.tolist())
print('first-block weights above diagonal:', e05_weights.triu(1).abs().max().item(),
      '| max |row sum - 1|:', (e05_weights.sum(-1) - 1).abs().max().item())
print('length-1 input logits:', tuple(model(X[:, :1]).shape))
try:
    model(documents)
except ValueError as error:
    e05_length_error = str(error)
    print('Expected failure:', e05_length_error)

In [ ]:
def shifted_loss(model):
    return F.cross_entropy(model(X).reshape(-1, V), y.reshape(-1))


def train(model, steps=200, lr=0.02):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.999),
                                  eps=1e-8, weight_decay=0.0)
    losses, gradient_norms = [], []
    for step in range(steps + 1):  # step 200 is evaluated but not applied
        optimizer.zero_grad(set_to_none=True)
        loss = shifted_loss(model)
        loss.backward()
        losses.append(loss.item())
        gradient_norms.append(torch.sqrt(sum(p.grad.square().sum() for p in model.parameters())).item())
        if step < steps:
            optimizer.step()
    return losses, gradient_norms


checkpoints = [0, 1, 5, 10, 20, 50, 100, 200]
model.load_state_dict(initial_state)
torch.manual_seed(7)
pre_losses, pre_gradient_norms = train(model)
with torch.no_grad():
    e05_probabilities = torch.softmax(model(X), dim=-1)
e05_predictions = e05_probabilities.argmax(dim=-1)
e05_bos_probabilities = e05_probabilities[:, 0, [1, 2]]
for step in checkpoints:
    print(f'step {step:3d}: loss {pre_losses[step]:.4f} | gradient norm {pre_gradient_norms[step]:.4f}')
print('floor:', round(loss_floor, 4), '| predictions:',
      [[vocabulary[i] for i in row] for row in e05_predictions.tolist()])
print('P(red), P(blue) after BOS:', e05_bos_probabilities.tolist())

**Check:** (a) `(2, 4, 7)`. (b) Only position 3; the prefix logits are **exactly** unchanged. (c) Positions 0–2 receive nonzero gradient and position 3 receives exactly zero. (d) Without the mask, earlier logits change and position 3 receives gradient. In the first block, every weight above the diagonal is $0$ and each row sums to $1$ up to float32 rounding, because masking happens before softmax. Position 0 has one allowed key, so its weight on itself is exactly $1$. A length-1 input works; a length-5 input is rejected because only four learned positions exist.

In the reference run, the loss starts near the uniform value $\log 7\approx1.95$ and ends **below 0.20** after 200 AdamW updates; it cannot go below the floor $\log(2)/4\approx0.1733$. It is not monotone. It can pause near $\log(2)/2\approx0.347$: that is the loss when the targets after `key`, like those after `BOS`, are still split $1/2$–$1/2$ between `opens` and `closes`, as in a bigram model. Leaving that plateau requires attention to read the earlier colour token. Transient loss spikes can occur with this fixed learning rate. The six deterministic transitions are predicted correctly in that run. Another platform can exit the plateau at a different update; report the observed loss if it has not reached the fitting criterion yet. After `BOS`, `red` and `blue` each receive probability close to $1/2$; the argmax at that position can be either, so top-1 accuracy there is not meaningful. This **fits one tiny batch**: it is a debugging check, not evidence of generalization.

## Pre-LN versus post-LN on the same toy batch

Change **only** the block normalization order. Both models use the same configuration, the same copied initial `state_dict`, the final LayerNorm, the toy batch, AdamW settings, and 200 full-batch updates, with no dropout or gradient clipping. We record the loss and global gradient norm **before** each update at selected steps.

In [ ]:
post_model = DecoderLM(norm_first=False)
post_model.load_state_dict(initial_state)
torch.manual_seed(7)
post_losses, post_gradient_norms = train(post_model)
with torch.no_grad():
    post_predictions = post_model(X).argmax(dim=-1)
print('step | pre-LN loss, grad norm | post-LN loss, grad norm')
for step in checkpoints:
    print(f'{step:4d} | {pre_losses[step]:.4f}, {pre_gradient_norms[step]:.4f}'
          f' | {post_losses[step]:.4f}, {post_gradient_norms[step]:.4f}')

**Reading the comparison:** report what this run shows: one seed, one tiny batch, one learning rate, two blocks. Both runs are bounded below by the same floor, and either may pause near $\log(2)/2$ for a different number of steps. Compare when each run leaves that plateau, the final loss, and the gradient norms, without choosing a winner in advance. A difference here does not establish a general ranking. [Xiong et al. (2020)](https://proceedings.mlr.press/v119/xiong20b.html) study deeper models, where post-LN typically needs learning-rate warm-up; our tiny model does not test that setting. Extended optimizer schedules belong to Lecture 06.

## Readings and continuation

- [Bahdanau et al. (2015)](https://arxiv.org/abs/1409.0473) and [Luong et al. (2015)](https://aclanthology.org/D15-1166/), §3: alignment in recurrent encoder–decoder models.

- [Vaswani et al. (2017)](https://arxiv.org/abs/1706.03762), §§3.1–3.5: architecture, multi-head attention, FFN, embeddings, and positions.
- [Xiong et al. (2020)](https://proceedings.mlr.press/v119/xiong20b.html): pre-LN and post-LN placement.
- [Devlin et al. (2019), §§3 and 3.1](https://aclanthology.org/N19-1423/): bidirectional encoders and masked prediction.
- [Su et al., RoFormer, §3](https://arxiv.org/abs/2104.09864): rotary position embeddings.
- [Ba et al. (2016)](https://arxiv.org/abs/1607.06450) and [Press and Wolf (2017)](https://arxiv.org/abs/1608.05859): LayerNorm and output-weight tying.

Lecture 06 continues with training the model: optimizer schedules, checkpoints, and decoding.